In [1]:
!pip uninstall mysqlclient mysql-connector mysql-connector-python-rf -y


In [2]:
pip install mysql-connector-python==8.0.33


Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd
import numpy as np
import mysql.connector

In [4]:
conn = mysql.connector.connect(
    host = "localhost",
    user = "root",
    password = "Satya@123",
    database = "dummy_project"
)

In [5]:
conn.is_connected()

True

In [6]:
#we will create an function out of it
def connect_to_db():
    return mysql.connector.connect(
    host = "localhost",
    user = "root",
    password = "Satya@123",
    database = "dummy_project"
)

In [9]:
connect_to_db().is_connected()

True

In [7]:
db = connect_to_db()
cursor = db.cursor(dictionary= True)

In [8]:
cursor

In [11]:
cursor.execute("select count(*) as total_suppliers from suppliers") 

In [12]:
row = cursor.fetchone()

In [20]:
list(row.values())[0]

50

In [13]:
def get_basic_info(cursor):
    queries = {
        "Total Suppliers": "SELECT COUNT(*) AS count FROM suppliers",

        "Total Products": "SELECT COUNT(*) AS count FROM products",

        "Total Categories Dealing": "SELECT COUNT(DISTINCT category) AS count FROM products",

        "Total Sale Value (Last 3 Months)": """
        SELECT ROUND(SUM(ABS(se.change_quantity) * p.price), 2) AS total_sale
        FROM stock_entries se
        JOIN products p ON se.product_id = p.product_id
        WHERE se.change_type = 'Sale'
        AND se.entry_date >= (
        SELECT DATE_SUB(MAX(entry_date), INTERVAL 3 MONTH) FROM stock_entries)
        """,

        "Total Restock Value (Last 3 Months)": """
        SELECT ROUND(SUM(se.change_quantity * p.price), 2) AS total_restock
        FROM stock_entries se
        JOIN products p ON se.product_id = p.product_id
        WHERE se.change_type = 'Restock'
        AND se.entry_date >= (
        SELECT DATE_SUB(MAX(entry_date), INTERVAL 3 MONTH) FROM stock_entries)
        """,

        "Below Reorder & No Pending Reorders": """
        SELECT COUNT(*) AS below_reorder
        FROM products p
        WHERE p.stock_quantity < p.reorder_level
        AND p.product_id NOT IN (
        SELECT DISTINCT product_id FROM reorders WHERE status = 'Pending')
        """
    }


In [16]:
get_basic_info(cursor)